# Learning 02: LLM-Driven Classification Dataset Generation

**Domain**: Cybersecurity threat intelligence

GLiClass training data format:
```json
{"text": "...", "all_labels": ["ransomware", "phishing", "ddos"], "true_labels": ["ransomware"]}
```

The key challenge: each example must include both **true labels** and **false labels** as `all_labels`.
The LLM generates both, giving the model negative examples to learn from.

## Attack type labels
| Label | Description |
|-------|-------------|
| `ransomware` | File encryption + ransom demand |
| `phishing` | Social engineering / fake login pages |
| `apt_attack` | Advanced persistent threat / nation-state |
| `ddos` | Distributed denial-of-service |
| `data_breach` | Unauthorized data exfiltration |
| `supply_chain_attack` | Compromised software/build pipeline |
| `zero_day` | Exploitation of unpatched vulnerability |

In [ ]:
import json, os
from openai import OpenAI

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "raw_security_reports.json")) as f:
    reports = json.load(f)

api_key = "your-api-key-here"  # Replace with your actual OpenAI key
client = OpenAI(api_key=api_key)

ALL_ATTACK_LABELS = [
    "ransomware", "phishing", "apt_attack", "ddos",
    "data_breach", "supply_chain_attack", "zero_day"
]
print(f"✓ Loaded {len(reports)} reports")
print(f"Classification labels: {ALL_ATTACK_LABELS}")

## 1. Classification Prompt with JSON Output

In [2]:
CLASSIFICATION_SYSTEM_PROMPT = """You are a cybersecurity analyst. Classify the given report.

Available attack type labels:
ransomware, phishing, apt_attack, ddos, data_breach, supply_chain_attack, zero_day

Return a JSON object with:
- "true_labels": list of labels that APPLY to this report (1-3 labels)
- "false_labels": list of 2-3 labels that clearly do NOT apply

Return ONLY the JSON object, nothing else."""


def call_llm_cls(client, text, model="gpt-4o-mini"):
    """Call LLM for classification. Returns parsed JSON dict."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ],
        temperature=0,
        response_format={"type": "json_object"}  # Force valid JSON
    )
    return json.loads(response.choices[0].message.content)


result = call_llm_cls(client, reports[0]['text'])
print(f"Text: {reports[0]['text'][:100]}...")
print(f"\nLLM output: {result}")

Text: The LockBit 3.0 ransomware group exploited CVE-2023-4966 in Citrix NetScaler ADC to gain initial acc...

LLM output: {'true_labels': ['ransomware', 'apt_attack', 'zero_day'], 'false_labels': ['phishing', 'ddos', 'data_breach']}


## 2. Build GLiClass Training Example

In [3]:
def build_gliclass_example(client, text):
    """
    Annotate a single text for GLiClass training.
    
    Returns:
        dict with 'text', 'all_labels', 'true_labels' keys
    """
    data = call_llm_cls(client, text)
    
    true_labels = [l for l in data.get('true_labels', []) if l in ALL_ATTACK_LABELS]
    false_labels = [l for l in data.get('false_labels', []) if l in ALL_ATTACK_LABELS]
    
    # Combine into all_labels (no duplicates)
    all_labels = list(set(true_labels + false_labels))
    
    if not true_labels or not all_labels:
        return None
    
    return {
        "text": text,
        "all_labels": all_labels,
        "true_labels": true_labels
    }


for i in range(3):
    ex = build_gliclass_example(client, reports[i]['text'])
    print(f"\nReport {i+1}:")
    print(f"  text:        {reports[i]['text'][:70]}...")
    print(f"  all_labels:  {ex['all_labels']}")
    print(f"  true_labels: {ex['true_labels']}")


Report 1:
  text:        The LockBit 3.0 ransomware group exploited CVE-2023-4966 in Citrix Net...
  all_labels:  ['ransomware', 'data_breach', 'phishing', 'zero_day', 'apt_attack', 'ddos']
  true_labels: ['ransomware', 'apt_attack', 'zero_day']



Report 2:
  text:        Security researchers identified a spear-phishing campaign targeting fi...
  all_labels:  ['ransomware', 'data_breach', 'phishing', 'ddos']
  true_labels: ['phishing']



Report 3:
  text:        APT29, also known as Cozy Bear, leveraged a zero-day vulnerability CVE...
  all_labels:  ['ransomware', 'phishing', 'zero_day', 'apt_attack', 'ddos']
  true_labels: ['apt_attack', 'zero_day']


## 3. Batch Annotation and Quality Checks

In [4]:
from tqdm import tqdm
from collections import Counter


def build_cls_dataset(client, reports, max_examples=None):
    """Build a complete GLiClass training dataset from raw reports."""
    dataset = []
    texts = [r['text'] for r in reports]
    if max_examples:
        texts = texts[:max_examples]
    
    for text in tqdm(texts, desc="Annotating classification"):
        try:
            ex = build_gliclass_example(client, text)
            if ex:
                dataset.append(ex)
        except Exception as e:
            print(f"  Error: {e}")
    return dataset


cls_dataset = build_cls_dataset(client, reports, max_examples=5)

print(f"\n✓ Generated {len(cls_dataset)} classification examples")

# Stats
label_counts = Counter(l for ex in cls_dataset for l in ex['true_labels'])
print("\nTrue label distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:25}: {count}")

Annotating classification:   0%|          | 0/5 [00:00<?, ?it/s]

Annotating classification:  20%|██        | 1/5 [00:01<00:05,  1.38s/it]

Annotating classification:  40%|████      | 2/5 [00:02<00:04,  1.45s/it]

Annotating classification:  60%|██████    | 3/5 [00:04<00:02,  1.30s/it]

Annotating classification:  80%|████████  | 4/5 [00:05<00:01,  1.19s/it]

Annotating classification: 100%|██████████| 5/5 [00:06<00:00,  1.29s/it]

Annotating classification: 100%|██████████| 5/5 [00:06<00:00,  1.30s/it]


✓ Generated 5 classification examples

True label distribution:
  ransomware               : 2
  apt_attack               : 2
  zero_day                 : 2
  phishing                 : 2
  ddos                     : 1
  data_breach              : 1


In [5]:
def validate_cls_dataset(dataset, valid_labels):
    """Validate GLiClass dataset format."""
    errors = []
    for i, ex in enumerate(dataset):
        if 'text' not in ex or 'all_labels' not in ex or 'true_labels' not in ex:
            errors.append(f"Example {i}: missing required keys")
            continue
        if not ex['true_labels']:
            errors.append(f"Example {i}: empty true_labels")
        if not set(ex['true_labels']).issubset(set(ex['all_labels'])):
            errors.append(f"Example {i}: true_labels not in all_labels")
        for label in ex['all_labels']:
            if label not in valid_labels:
                errors.append(f"Example {i}: unknown label '{label}'")
    return errors


errors = validate_cls_dataset(cls_dataset, ALL_ATTACK_LABELS)
if errors:
    print(f"❌ Errors: {errors}")
else:
    print("✓ All examples valid")

# Save
output_path = os.path.normpath(os.path.join(fixtures, "..", "expected", "cls_dataset_sample.json"))
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(cls_dataset, f, indent=2)
print(f"✓ Saved cls_dataset_sample.json")

✓ All examples valid
✓ Saved cls_dataset_sample.json
